# OpenEvolve TSP Demo —— 從頭跑到尾(教學版)

這個 notebook 把**環境檢查 → 安裝 → 測試 → 演化 → 看結果**全部做成一格一格,
你只要由上往下逐格按 ▶ 執行即可,不用切到終端機。

> **開始前**:用 VS Code 右上角選好 Python 直譯器(Windows 原生那個,不要 WSL)。


## 第 0 步:檢查 Ollama 有沒有裝好、模型在不在

下面這格會列出本機已下載的 Ollama 模型。
- 看到 `qwen2.5-coder:7b` → 跳過第 1 步。
- 沒看到、或報錯 → 先做第 1 步下載模型。


In [ ]:
!ollama list

## 第 1 步:下載模型(只需做一次,約 4.7GB)

**如果上一步已經看到 `qwen2.5-coder:7b`,這格可以跳過。**
第一次下載會比較久,請耐心等到出現 `success`。


In [ ]:
!ollama pull qwen2.5-coder:7b

## 第 2 步:安裝 Python 套件

安裝 OpenEvolve 與繪圖套件(matplotlib)。


In [ ]:
!pip install -r requirements.txt -q
print("套件安裝完成")

## 第 2.5 步:確認連得到 Ollama 服務

OpenEvolve 要透過 `localhost:11434` 連本機的 Ollama。
下面這格測試連線 —— 成功會印出 ✅。
失敗的話,開一個終端機執行 `ollama serve` 放著別關,再重跑這格。


In [ ]:
import urllib.request
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5) as r:
        print("✅ Ollama 服務正常,連得到 localhost:11434")
except Exception as e:
    print("❌ 連不到 Ollama。請開終端機執行  ollama serve  再重跑這格。")
    print("   錯誤:", e)

## 第 3 步:先測評分(不燒 LLM,確認檔案沒問題)

用初始程式(最近鄰法)求一條路線並評分,順便畫出來看。
**預期**:分數約 `0.75`,而且看到一條**亂繞、有交叉**的初始路線。


In [ ]:
import matplotlib.pyplot as plt
from evaluator import CITIES, _tour_length, evaluate
from initial_program import solve

import os
result = evaluate(os.path.join(os.getcwd(), "initial_program.py"))
print("評分結果:", result)

tour = solve(CITIES)
length = _tour_length(tour)

def plot_tour(tour, title):
    xs = [CITIES[i][0] for i in tour] + [CITIES[tour[0]][0]]
    ys = [CITIES[i][1] for i in tour] + [CITIES[tour[0]][1]]
    plt.figure(figsize=(6, 6))
    plt.plot(xs, ys, "-o", linewidth=1.5, markersize=6)
    plt.title(title)
    plt.show()

plot_tour(tour, f"初始路線(最近鄰法)— 總長 {length:.1f}")

## 第 4 步:正式啟動演化

讓本地 LLM 一代一代改寫求解程式。每一代會印出分數(`combined_score`,越高越好)。

> ⏳ 視顯卡而定,30 代大約數分鐘。中途 `route.png` 會即時更新。


In [ ]:
from openevolve import run_evolution

HERE = os.getcwd()
result = run_evolution(
    initial_program=os.path.join(HERE, "initial_program.py"),
    evaluator=os.path.join(HERE, "evaluator.py"),
    config=os.path.join(HERE, "config.yaml"),
)
print("\n✅ 演化完成!")

## 第 5 步:看演化後的最佳路線,跟初始比較

**預期**:總長度變短,路線變成乾淨漂亮的環、交叉消失。


In [ ]:
import importlib.util

best_path = os.path.join(HERE, "openevolve_output", "best", "best_program.py")
spec = importlib.util.spec_from_file_location("best", best_path)
best = importlib.util.module_from_spec(spec)
spec.loader.exec_module(best)

best_tour = best.solve(CITIES)
best_length = _tour_length(best_tour)

print(f"初始路線長度:{length:.1f}")
print(f"演化後長度:  {best_length:.1f}")
print(f"縮短了:      {length - best_length:.1f}  ({(1 - best_length/length)*100:.1f}%)")

plot_tour(best_tour, f"演化後最佳路線 — 總長 {best_length:.1f}")

## 想再深入?

- 打開 `openevolve_output/best/best_program.py`,看 **LLM 加了什麼技巧**(常見:2-opt、模擬退火、多起點重啟)。
- 改 `config.yaml` 的 `max_iterations` 跑更多代,看能不能更短。
- 顯卡 20GB+ 的話,把模型換成 `qwen2.5-coder:14b`(記得先 `ollama pull`),演化品質更好。

---

### 常見問題
| 症狀 | 解法 |
|---|---|
| 第 4 步卡住沒分數 | Ollama 沒開 → 終端機 `ollama serve` |
| `No valid diffs found` | 7B 偶爾不照格式,多跑幾代通常恢復 |
| 分數一直 0 | 演化出的程式暫時跑爛,幾代後會自己修正 |
